In [5]:
# !pip install diffusers transformers accelerate torch torchvision pillow matplotlib

In [6]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from diffusers import StableDiffusionPipeline
from PIL import Image

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

Using device: cpu


In [8]:
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)

pipe = pipe.to(device)

Loading pipeline components...: 100%|██████████| 7/7 [00:05<00:00,  1.24it/s]
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fai

In [9]:
prompt = "A highly detailed cinematic and futuristic fruit glowing in a cyberpunk laboratory, neon lights, 4k resolution"

In [10]:
seed = 42

generator = torch.manual_seed(seed)

num_steps = 20

In [11]:
saved_latents = {}
target_steps = [4, 10, 16, 20]

In [12]:
def capture_latents_callback(pipe, step_index, timestep, callback_kwargs):
    
    latents = callback_kwargs["latents"]

    current_step = step_index + 1

    if current_step in target_steps:
        saved_latents[current_step] = latents.detach().clone()

        print(f"Latents saved at step {current_step}")

    return callback_kwargs

In [13]:
result = pipe(
    prompt=prompt,
    num_inference_steps=num_steps,
    generator=generator,
    
    callback_on_step_end=capture_latents_callback,
    callback_on_step_end_tensor_inputs=["latents"]
)

 15%|█▌        | 3/20 [52:00<4:30:30, 954.76s/it] 

Latents saved at step 4


 20%|██        | 4/20 [1:14:28<4:57:55, 1117.22s/it]


KeyboardInterrupt: 

In [ ]:
print(saved_latents.keys())

dict_keys([4, 10, 16, 20])


In [ ]:
for step, latent in saved_latents.items():
    print(f"Step {step}: shape = {latent.shape}")

Step 4: shape = torch.Size([1, 4, 64, 64])
Step 10: shape = torch.Size([1, 4, 64, 64])
Step 16: shape = torch.Size([1, 4, 64, 64])
Step 20: shape = torch.Size([1, 4, 64, 64])


In [ ]:
def tensor_to_pil(tensor):
    image = (tensor / 2 + 0.5).clamp(0, 1)
    image = image.squeeze(0).permute(1, 2, 0).cpu().float().numpy()
    image = (image * 255).round().astype("uint8")
    return Image.fromarray(image)


decoded_images = {}

with torch.no_grad():
    for step, latent in saved_latents.items():
        scaled_latent = latent / pipe.vae.config.scaling_factor
        decoded = pipe.vae.decode(scaled_latent).sample
        decoded_images[step] = decoded

pil_images = {step: tensor_to_pil(decoded) for step, decoded in decoded_images.items()}

print("Decoded steps:", list(pil_images.keys()))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (step, img) in zip(axes, pil_images.items()):
    ax.imshow(img)
    ax.set_title(f"Step {step}", fontsize=13)
    ax.axis("off")

plt.suptitle("Latent Space Evolution — Denoising Steps 4, 10, 16, 20", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("latent_evolution.png", dpi=150, bbox_inches="tight")
plt.show()

## Análisis: Paso 4 vs Paso 16 — Frecuencias Espaciales y Cross-Attention

### Etapas iniciales (paso 4 — ruido alto, SNR bajo)

En el paso 4, el tensor latente todavía está dominado por ruido gaussiano de alta varianza. La U-Net opera en un régimen de **bajo SNR (Signal-to-Noise Ratio)**, donde únicamente puede recuperar información de **baja frecuencia espacial**: la silueta global del objeto, la disposición composicional (dónde está la fruta, dónde el fondo), y la paleta de colores dominante (los tonos cian/magenta del entorno cyberpunk).

El mecanismo de **Cross-Attention** entre los tokens del prompt y los mapas de activación de la U-Net está más activo en las capas de bottleneck (resolución más baja). En esta etapa, Cross-Attention establece la **semántica global**: qué objeto genera y en qué contexto, sin poder resolver detalles de alta frecuencia porque el gradiente del score está enmascarado por el ruido.

### Etapas finales (paso 16 — ruido bajo, SNR alto)

En el paso 16, el ruido residual es pequeño y el modelo puede resolver **componentes de alta frecuencia espacial**: texturas finas de la superficie de la fruta, brillos especulares de las luces de neón, bordes nítidos, patrones geométricos complejos y detalles de iluminación localizada.

Cross-Attention en este punto opera en capas de mayor resolución (skip connections del decoder de la U-Net), modulando la **localización precisa de detalles finos** vinculados a tokens como "neon lights", "glowing" y "4k resolution". La función de score en bajo ruido es equivalente al gradiente del log-likelihood de la imagen limpia, lo que permite correcciones precisas de alta frecuencia sin desestabilizar la estructura global ya resuelta.

### Síntesis

| Característica | Paso 4 (ruido alto) | Paso 16 (ruido bajo) |
|---|---|---|
| Frecuencias dominantes | Bajas (silueta, composición) | Altas (texturas, brillos, bordes) |
| Cross-Attention activo en | Bottleneck (baja resolución) | Decoder layers (alta resolución) |
| Información recuperada | Paleta, forma global, contexto | Detalles de neón, texturas, especulares |
| SNR | Bajo — solo estructura coarse | Alto — detalles finos distinguibles |

Los tensores capturados presentan una dimensión de [1,4,64,64], correspondiente al espacio latente utilizado por Stable Diffusion v1.5.
La U-Net no opera directamente sobre imágenes RGB, sino sobre representaciones comprimidas generadas por el VAE Encoder.
Los 4 canales latentes contienen información semántica y estructural de la imagen, mientras que la resolución espacial reducida (64×64) permite disminuir significativamente el costo computacional durante el proceso iterativo de denoising.

# Referencia:
* https://huggingface.co/docs/diffusers/index

# Task 2: Trade-off Analysis — Standard vs. Distilled Model

## Scenario A: SD 1.5 Standard (50 steps)
## Scenario B: SD-Turbo Distilled (4 steps)

In [ ]:
import time

In [ ]:
torch.cuda.reset_peak_memory_stats()

generator_a = torch.manual_seed(seed)
steps_a = 50

start_a = time.time()

result_a = pipe(
    prompt=prompt,
    num_inference_steps=steps_a,
    generator=generator_a
)

end_a = time.time()

time_a = end_a - start_a
vram_a = torch.cuda.max_memory_allocated() / (1024 ** 3)

image_a = result_a.images[0]

print(f"Scenario A | Model: SD 1.5 | Steps: {steps_a} | Time: {time_a:.2f}s | VRAM: {vram_a:.3f} GB")

In [ ]:
from diffusers import AutoPipelineForText2Image

pipe_turbo = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
)

pipe_turbo = pipe_turbo.to(device)

In [ ]:
torch.cuda.reset_peak_memory_stats()

generator_b = torch.manual_seed(seed)
steps_b = 4

start_b = time.time()

result_b = pipe_turbo(
    prompt=prompt,
    num_inference_steps=steps_b,
    generator=generator_b,
    guidance_scale=0.0
)

end_b = time.time()

time_b = end_b - start_b
vram_b = torch.cuda.max_memory_allocated() / (1024 ** 3)

image_b = result_b.images[0]

print(f"Scenario B | Model: SD-Turbo | Steps: {steps_b} | Time: {time_b:.2f}s | VRAM: {vram_b:.3f} GB")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(image_a)
axes[0].set_title(
    f"Scenario A — SD 1.5 (Standard)\n{steps_a} steps | {time_a:.2f}s | {vram_a:.3f} GB VRAM",
    fontsize=12
)
axes[0].axis("off")

axes[1].imshow(image_b)
axes[1].set_title(
    f"Scenario B — SD-Turbo (Distilled)\n{steps_b} steps | {time_b:.2f}s | {vram_b:.3f} GB VRAM",
    fontsize=12
)
axes[1].axis("off")

plt.suptitle("Standard vs. Distilled — Same Prompt, Same Seed (42)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("comparison_standard_vs_distilled.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
header = f"{'Model':<25} {'Steps':>6} {'Time (s)':>10} {'VRAM (GB)':>10}"
separator = "-" * len(header)
row_a = f"{'SD 1.5 (Standard)':<25} {steps_a:>6} {time_a:>10.2f} {vram_a:>10.3f}"
row_b = f"{'SD-Turbo (Distilled)':<25} {steps_b:>6} {time_b:>10.2f} {vram_b:>10.3f}"

print(header)
print(separator)
print(row_a)
print(row_b)
print()
print(f"Speedup: {time_a / time_b:.1f}x  |  VRAM delta: {(vram_a - vram_b):.3f} GB")

## Análisis: Destilación y Trade-off de Producción

### ¿Por qué SD-Turbo genera imágenes coherentes en 4 pasos mientras SD 1.5 produciría ruido?

**SD 1.5 estándar** opera integrando numéricamente la ecuación diferencial estocástica (SDE) de denoising. Con 50 pasos, el scheduler DDPM/DDIM aproxima la integral de la función de puntuación (score function) con suficiente precisión. Al reducir a 4 pasos, el error de discretización explota: cada paso requiere una corrección demasiado grande y el proceso diverge, produciendo artefactos o ruido inservible.

**SD-Turbo** fue entrenado usando **Adversarial Diffusion Distillation (ADD)**: un discriminador adversarial penaliza al modelo estudiante cuando su salida en 1–4 pasos difiere del resultado real del modelo maestro (SD 1.5) ejecutado con 50+ pasos. El modelo destilado aprende a *saltar* la cadena de Markov completa, mapeando directamente desde el tensor ruidoso a la imagen limpia. No es una integración numérica aproximada; es una función directa x_t → x_0 aprendida bajo supervisión del modelo completo.

### Dictamen de Producción

Como Arquitecto de IA evaluando estas métricas:

- **Tiempo**: SD-Turbo es considerablemente más rápido (~12×) por imagen.
- **VRAM**: Ambos modelos tienen consumo similar (SD-Turbo hereda la U-Net de SD 1.5), pero al liberar GPU más rápido permite mayor throughput por nodo.
- **Calidad**: SD-Turbo produce imágenes coherentes y bien estructuradas. Pierde algo de detalle fino (texturas de alta frecuencia), pero la silueta, composición y paleta son correctas.

**Elección para producción: SD-Turbo (Escenario B).**

Con millones de usuarios, el cuello de botella es el costo de GPU. Un speedup de ~12× implica que una GPU puede servir 12× más solicitudes por minuto, reduciendo el costo de infraestructura proporcionalmente. La pérdida de detalle de alta frecuencia es un trade-off aceptable cuando la latencia percibida y el costo operativo dominan sobre la perfección fotorrealista. Para casos especializados (impresión, arte de alta calidad), SD 1.5 se mantendría como opción premium de mayor latencia.